# D091 — Object-Oriented Programming Principles

Basic OOP concepts explained with E-Comm terminologies, difference products types like physical and digital products and accepts different payment methods. These examples introduce the three core OOP principles:

1. **Encapsulation** — protect an object's valid state
2. **Inheritance** — create a specialized class from an existing class
3. **Polymorphism** — use different objects through the same interface

# 1. Encapsulation

Encapsulation keeps data and the operations that control it inside one class. Its purpose is not merely to hide data; it prevents an object from entering an invalid state.

For example, QuickCart should not allow a product to have a negative price. Instead of letting every part of the application validate prices independently, the `Product` class owns that rule.

## Python has no enforced access specifiers

Python does not have Java/C++ keywords such as `public`, `protected`, and `private`. It uses naming conventions:

| Form | Meaning |
|---|---|
| `name` | Public API; normal access is expected |
| `_name` | Internal/protected-by-convention; external code should avoid direct access |
| `__name` | Name-mangled to reduce accidental access and subclass name clashes |

A leading underscore is a request to programmers, not a security boundary. Double underscores trigger **name mangling**: `__price` is stored with a name similar to `_Product__price`. It can still be reached deliberately, so it is not truly private.

In [ ]:
class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id  # Public
        self._name = name             # Internal by convention
        self.__price = float(price)   # Name-mangled

    def display_price(self):
        return f"₹{self.__price:,.2f}"


product = Product("P101", "Smartphone", 29999)

print(product.product_id)
print(product._name)          # Possible, but external code should avoid it
print(product.display_price())
print(product.__dict__)       # Shows the mangled storage name

## Properties: attribute syntax with getter and setter logic

Python usually avoids methods such as `get_price()` and `set_price()`. A **property** keeps simple attribute syntax while automatically calling controlled getter and setter methods.

```python
product.price        # calls the method decorated with @property
product.price = 500  # calls the method decorated with @price.setter
```

This allows validation to be added without changing how callers use the attribute.

In [ ]:
class Product:
    def __init__(self, product_id, name, price, stock=0):
        self.product_id = product_id
        self.name = name
        self.price = price  # Calls the setter during initialization
        self.stock = stock

    @property
    def price(self):
        """Return the current product price."""
        return self._price

    @price.setter
    def price(self, value):
        """Validate and store a product price."""
        value = float(value)
        if value <= 0:
            raise ValueError("price must be greater than zero")
        self._price = value


product = Product("P102", "Laptop", 64999, stock=12)

print(product.price)  # Getter runs
product.price = 61999 # Setter runs
print(product.price)

try:
    product.price = -100
except ValueError as error:
    print(error)

## Read-only and computed properties

A property without a setter is read-only through normal attribute assignment. A property can also calculate a value instead of storing it.

In [ ]:
class CartItem:
    def __init__(self, product_name, unit_price, quantity):
        self.product_name = product_name
        self.unit_price = float(unit_price)
        self.quantity = quantity

    @property
    def line_total(self):
        """Compute the total; no separate stored value is needed."""
        return self.unit_price * self.quantity


item = CartItem("Wireless Mouse", 1499, 2)
print(item.line_total)

## Dynamic attribute access: `getattr` and `setattr`

`getattr(object, name)` reads an attribute whose name is stored as a string. `setattr(object, name, value)` assigns one. These built-ins are useful for dynamic configuration and imported data.

They still use the normal attribute mechanism. Therefore, `setattr(product, "price", value)` calls the `price` property setter and retains its validation.

In [ ]:
product = Product("P103", "Mechanical Keyboard", 3499, stock=25)

attribute_name = "price"
print(getattr(product, attribute_name))

setattr(product, attribute_name, 3299)  # Calls the price setter
print(product.price)

# A default avoids AttributeError when an optional attribute is absent.
print(getattr(product, "brand", "Brand not provided"))

### Encapsulation summary

- Keep validation and state-changing operations inside the class.
- Use public attributes when unrestricted access is safe.
- Use `_name` for internal implementation details.
- Use properties when reading or assigning an attribute requires logic.
- Do not use double underscores as a security mechanism.

# 2. Inheritance

Inheritance creates a specialized class from an existing class.

- The existing class is the **base class**, **parent class**, or **superclass**.
- The specialized class is the **derived class**, **child class**, or **subclass**.
- A derived object receives the base class's accessible methods and attributes and may add or replace behaviour.

Inheritance models an **is-a** relationship: a `PhysicalProduct` is a `Product`. For a **has-a** relationship—an `Order` has products—composition is normally more appropriate.

## Calling the base-class constructor with `super()`

If a derived class defines `__init__`, it must initialize the base-class portion of the object. `super().__init__(...)` calls the base-class constructor according to Python's method resolution order.

Pass the common product parameters to `super()` and initialize only specialized attributes in the derived constructor. This avoids duplicating base-class initialization logic.

In [ ]:
class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id
        self.name = name
        self.price = float(price)

    def delivery_message(self):
        return "Delivery information is not available."


class PhysicalProduct(Product):
    def __init__(self, product_id, name, price, weight_kg):
        super().__init__(product_id, name, price)
        self.weight_kg = float(weight_kg)

    def delivery_message(self):
        return f"{self.name} will be shipped to the delivery address."


class DigitalProduct(Product):
    def __init__(self, product_id, name, price, download_size_mb):
        super().__init__(product_id, name, price)
        self.download_size_mb = download_size_mb

    def delivery_message(self):
        return f"A download link for {self.name} will be sent by email."


monitor = PhysicalProduct("P201", "27-inch Monitor", 18999, 5.4)
ebook = DigitalProduct("P202", "Python Handbook", 799, 18)

print(monitor.product_id, monitor.weight_kg)
print(ebook.product_id, ebook.download_size_mb)

## Inherited and overridden methods

`PhysicalProduct` and `DigitalProduct` inherit the attributes initialized by `Product.__init__`.

Both derived classes **override** `delivery_message` by defining a method with the same name. Python selects the method using the object's actual type.

In [ ]:
print(monitor.delivery_message())
print(ebook.delivery_message())

print(isinstance(monitor, PhysicalProduct))
print(isinstance(monitor, Product))
print(issubclass(DigitalProduct, Product))

### Inheritance summary

- Put genuinely shared state and behaviour in the base class.
- Use `super().__init__(...)` to initialize the base-class part of a derived object.
- Pass base-class parameters through the derived constructor explicitly.
- Override a method only when the derived type needs specialized behaviour.
- Prefer composition when the relationship is **has-a**, not **is-a**.
- Avoid deep inheritance hierarchies; they make behaviour harder to trace.

# 3. Polymorphism

Polymorphism means “many forms.” Different object types respond to the same operation in their own way.

The code using the objects calls one common method and does not need a separate `if` statement for every type. Method overriding is one source of polymorphism.

In [ ]:
def show_delivery_instructions(products):
    """Use every product through the same method interface."""
    for product in products:
        print(product.delivery_message())


cart_products = [
    PhysicalProduct("P301", "Mechanical Keyboard", 3499, 0.9),
    DigitalProduct("P302", "Data Engineering Course", 1999, 850),
]

show_delivery_instructions(cart_products)

## Duck typing: Python's practical polymorphism

Python often uses **duck typing**: if an object provides the required method, it can be used—its inheritance tree does not have to be checked first.

The function below accepts any payment object that supplies a callable `pay(amount)` method. `CardPayment` and `UpiPayment` share an interface without requiring inheritance.

In [ ]:
class CardPayment:
    def __init__(self, last_four_digits):
        self.last_four_digits = last_four_digits

    def pay(self, amount):
        return f"Paid ₹{amount:,.2f} using card ending {self.last_four_digits}."


class UpiPayment:
    def __init__(self, upi_id):
        self.upi_id = upi_id

    def pay(self, amount):
        return f"Paid ₹{amount:,.2f} using UPI ID {self.upi_id}."


def complete_payment(payment_method, amount):
    return payment_method.pay(amount)


card = CardPayment("4242")
upi = UpiPayment("customer@bank")

print(complete_payment(card, 2499))
print(complete_payment(upi, 2499))

## Final recap

| Principle | Main idea | QuickCart example |
|---|---|---|
| Encapsulation | Control access and keep objects valid | The `price` property rejects invalid prices |
| Inheritance | Reuse and specialize an is-a relationship | Physical and digital products extend `Product` |
| Polymorphism | One interface, type-specific behaviour | Each product delivers differently; each payment type implements `pay` |

Good object-oriented design does not mean using inheritance everywhere. Start with clear responsibilities, encapsulate important rules, use inheritance only for a genuine is-a relationship, and rely on small common interfaces for polymorphism.